In [1]:
import re
import numpy as np
import pandas as pd

from collections import defaultdict, Counter

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, confusion_matrix

# =========================
# CONFIG
# =========================
DATA_PATH = "../data/processed/v3/development_v3.csv"

RANDOM_STATE = 42
N_SPLITS = 5
USE_ONLY_FIRST_FOLD = True

# Rule mining params
MIN_TOTAL_FREQ = 80          # minimum occurrences in TRAIN to consider a token
MIN_PURITY = 0.92            # dominant_class_freq / total_freq in TRAIN
MAX_RULES_PER_CLASS = 60     # cap per class to avoid overfitting
RULE_TOKEN_RE = re.compile(r"[a-z0-9_]{3,}")  # conservative tokenization

# Rule application params (gating)
APPLY_RULE_IF_FOUND = True   # if True, rule can override model for those samples
RULE_PRIORITY = "best_purity_then_freq"  # how to resolve multiple hits

# =========================
# TEXT CLEANING (KEEP LINKS)
# =========================
def clean_keep_links(s: str) -> str:
	"""
	Cleans weird chars but keeps href/http tokens, domain pieces, etc.
	Does NOT remove tags aggressively: you keep variability.
	"""
	if s is None:
		return ""
	s = str(s)

	# Normalize whitespace
	s = s.replace("\u00a0", " ")
	s = re.sub(r"\s+", " ", s)

	# Remove non-printing control chars
	s = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", " ", s)

	# Keep basic punctuation that helps char-ngrams; remove only very weird symbols
	# (Do not strip <> or / or = because they carry HTML/link structure)
	s = re.sub(r"[«»„”¨•…]", " ", s)

	return s.strip()

def build_text(df: pd.DataFrame) -> pd.Series:
	# Title CAN help, but if your title is noisy in your version, keep it optional
	title = df["title"].fillna("").astype(str).map(clean_keep_links)
	article = df["article"].fillna("").astype(str).map(clean_keep_links)
	# concatenate with separator to help model learn boundaries
	return (title + " \n\n " + article).str.strip()

# =========================
# RULE MINING (TRAIN ONLY)
# =========================
def tokenize_for_rules(text: str):
	# lowercase to merge variants
	text = text.lower()
	return RULE_TOKEN_RE.findall(text)

def mine_pure_rules(texts: pd.Series, y: pd.Series):
	"""
	Find tokens that are highly class-pure in TRAIN.
	Returns:
	- rule_token_to_class: dict token -> class
	- rule_meta: dict token -> (purity, total_freq, dominant_freq)
	"""
	per_token_counts = defaultdict(lambda: np.zeros(len(np.unique(y)), dtype=int))
	classes = sorted(np.unique(y))
	class_to_idx = {c:i for i,c in enumerate(classes)}

	for txt, lab in zip(texts, y):
		# Use set() to avoid counting repeated token in same doc too much
		toks = set(tokenize_for_rules(txt))
		li = class_to_idx[lab]
		for t in toks:
			per_token_counts[t][li] += 1

	rule_token_to_class = {}
	rule_meta = {}

	per_class_rules = defaultdict(list)

	for token, arr in per_token_counts.items():
		total = int(arr.sum())
		if total < MIN_TOTAL_FREQ:
			continue
		dom_i = int(arr.argmax())
		dom_class = classes[dom_i]
		dom_freq = int(arr[dom_i])
		purity = dom_freq / total

		if purity >= MIN_PURITY:
			per_class_rules[dom_class].append((token, purity, total, dom_freq))

	# Keep only top rules per class (avoid exploding)
	for c, lst in per_class_rules.items():
		if RULE_PRIORITY == "best_purity_then_freq":
			lst.sort(key=lambda x: (x[1], x[2]), reverse=True)
		else:
			lst.sort(key=lambda x: x[2], reverse=True)

		lst = lst[:MAX_RULES_PER_CLASS]
		for token, purity, total, dom_freq in lst:
			rule_token_to_class[token] = c
			rule_meta[token] = (purity, total, dom_freq)

	return rule_token_to_class, rule_meta

def apply_rules(texts: pd.Series, rule_token_to_class: dict, rule_meta: dict):
	"""
	For each text, if any rule token appears, assign class by best rule.
	Returns:
	- rule_pred: np.array with class labels or -1 if no rule matched
	- matched_token: list token matched (or None)
	"""
	rule_pred = np.full(len(texts), -1, dtype=int)
	matched_token = [None] * len(texts)

	for i, txt in enumerate(texts):
		toks = set(tokenize_for_rules(txt))
		hits = [t for t in toks if t in rule_token_to_class]
		if not hits:
			continue

		if RULE_PRIORITY == "best_purity_then_freq":
			# choose token with highest purity then total freq
			hits.sort(key=lambda t: (rule_meta[t][0], rule_meta[t][1]), reverse=True)
		else:
			hits.sort(key=lambda t: rule_meta[t][1], reverse=True)

		best = hits[0]
		rule_pred[i] = int(rule_token_to_class[best])
		matched_token[i] = best

	return rule_pred, matched_token

# =========================
# MODEL (TEXT + SOURCE)
# =========================
def make_model():
	# NOTE: you can swap LogisticRegression with SGDClassifier(loss="log_loss") if faster
	pre = ColumnTransformer(
		transformers=[
			("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),
			("w_tfidf", TfidfVectorizer(
				analyzer="word",
				ngram_range=(1,2),
				min_df=3,
				max_df=0.9,
				sublinear_tf=True,
				max_features=250_000
			), "text"),
			("c_tfidf", TfidfVectorizer(
				analyzer="char_wb",
				ngram_range=(3,5),
				min_df=3,
				max_df=0.9,
				sublinear_tf=True,
				max_features=300_000
			), "text"),
		],
		remainder="drop",
		n_jobs=-1
	)

	clf = LogisticRegression(
		C=2.0,
		class_weight="balanced",  # you saw it helps macro F1
		max_iter=2000,
		n_jobs=-1
	)

	return Pipeline([
		("pre", pre),
		("clf", clf)
	])

# =========================
# RUN (1 FOLD)
# =========================
df = pd.read_csv(DATA_PATH)
df["source"] = df["source"].fillna("").astype(str)
df["text"] = build_text(df)

X = df[["source", "text"]]
y = df["label"].astype(int)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

for fold_id, (tr, te) in enumerate(skf.split(X, y), start=1):
	X_tr = X.iloc[tr].copy()
	y_tr = y.iloc[tr].copy()
	X_te = X.iloc[te].copy()
	y_te = y.iloc[te].copy()

	# ---- Mine rules ONLY on train
	rule_token_to_class, rule_meta = mine_pure_rules(X_tr["text"], y_tr)
	print(f"[Fold {fold_id}] Mined rules:", len(rule_token_to_class))

	# ---- Train model
	model = make_model()
	model.fit(X_tr, y_tr)

	# ---- Predict model
	proba = model.predict_proba(X_te)
	model_pred = proba.argmax(axis=1)

	# ---- Apply rules on test (no leakage because rules mined from train)
	rule_pred, matched_token = apply_rules(X_te["text"], rule_token_to_class, rule_meta)

	if APPLY_RULE_IF_FOUND:
		final_pred = model_pred.copy()
		mask = (rule_pred != -1)
		final_pred[mask] = rule_pred[mask]
	else:
		final_pred = model_pred

	# ---- Metrics
	macro = f1_score(y_te, final_pred, average="macro")
	print(f"\nFOLD {fold_id} MACRO F1: {macro:.6f}")

	print("\nConfusion Matrix:\n", confusion_matrix(y_te, final_pred))
	print("\nReport:\n", classification_report(y_te, final_pred, digits=3))

	# Optional: coverage of rules
	mask = (rule_pred != -1)
	print(f"\nRule coverage: {mask.mean():.3f}  (matched {mask.sum()} / {len(mask)})")
	if mask.any():
		print("Rule-only macro F1 on matched subset:",
			f1_score(y_te[mask], final_pred[mask], average="macro"))

	# Show a few most-used matched tokens
	counter = Counter([t for t in matched_token if t is not None])
	print("\nTop matched rule tokens:", counter.most_common(20))

	if USE_ONLY_FIRST_FOLD:
		break


[Fold 1] Mined rules: 59

FOLD 1 MACRO F1: 0.711671

Confusion Matrix:
 [[3350  147  136  288   45  649   93]
 [  83 1713  113   75   17   72   44]
 [  69  133 1842   75    6   51   56]
 [ 201  106  112 1129  124  255   69]
 [  19    9    4   60 1568   51    4]
 [ 536  131   65  321  131 1349   78]
 [  35   15   13   32    6   34  486]]

Report:
               precision    recall  f1-score   support

           0      0.780     0.712     0.744      4708
           1      0.760     0.809     0.784      2117
           2      0.806     0.825     0.816      2232
           3      0.570     0.566     0.568      1996
           4      0.827     0.914     0.868      1715
           5      0.548     0.517     0.532      2611
           6      0.586     0.783     0.670       621

    accuracy                          0.715     16000
   macro avg      0.697     0.732     0.712     16000
weighted avg      0.715     0.715     0.713     16000


Rule coverage: 0.100  (matched 1599 / 16000)
Rule-onl